In [2]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import ollama
from utils import downsample, squashOneHots, truncateDecimals
from sklearn.neighbors import NearestNeighbors

In [2]:
def write_prompt(X_train,y_train,X_test,header,filename="prompt.txt"):
    output_string = ""

    output_string += header + "\n\n"

    output_string += "Training data (X_train | y_train):\n"

    train_full = X_train
    train_full["class"] = y_train

    output_string += train_full.to_string() + "\n\n"

    # print augmented matrix
    output_string += "Unlabeled data (X_test):\n\n"
    
    output_string += X_test.to_string() + "\n\n"

    output_string += "Please proceed in generating a comma-separated list of classes corresponding to each entry in the Unlabeled Data, in order. Your output should be in the format 1,5,2,6,3,5,7,..."
    output_string += f"\nYou should output a total of {len(X_test)} comma-separated classes in the range [{1},{7}]."
    # note: this does not handle the actual input/response with the LLM! Just writes the prompt.

    file = open(f"./prompts/{filename}","w")
    file.write(output_string)
    file.close()


In [2]:
train,valid,test = pd.read_csv("data/train.csv"),pd.read_csv("data/valid.csv"),pd.read_csv("data/test.csv")

train = downsample(train, N=6000)
valid = downsample(valid, N=2000)
test = downsample(test, N=2000)

X_train, y_train = train.drop(columns="class"), train["class"]
X_valid, y_valid = valid.drop(columns="class"), valid["class"]
X_test, y_test = test.drop(columns="class"), test["class"]

In [5]:
header = "You are tasked with classifying trees into one of seven classes based on several geographic parameters, including elevation, aspect, slope, distance to hydrology, distance to roadways, and hillshade at various times of the day. Your task is to learn associations between these attributes and the seven possible output classes, and then try your best to assign class labels (y_test) to each of the unlabeled entries in the dataset (X_test)."

write_prompt(X_train,y_train,X_test,header)

In [6]:
test_prompt = open("./prompts/prompt.txt","r").read()

stream = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': test_prompt}],
    stream=True,
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)


KeyboardInterrupt: 

# Nearest Neighbors Shenanigans

In [3]:
# Gets nearest neighbors to "point" from among the "data" dataset, excluding class
# Uses weights for one-hot columns to avoid heavy weighting

def getNearestNeighbors(data, point, N=25, soil_weight=0.50, wilderness_weight=0.50):
    data = data.drop(columns=["class"],errors="ignore")
    point = point.drop(columns=["class"],errors="ignore")

    for i in range(1,40+1):
        data["Soil_Type"+str(i)] *= soil_weight
        point["Soil_Type"+str(i)] *= soil_weight

    for j in range(1,4+1):
        data["Wilderness_Area"+str(j)] *= wilderness_weight
        point["Wilderness_Area"+str(j)] *= wilderness_weight

    nn = NearestNeighbors(n_neighbors=N, metric="euclidean")
    nn.fit(data)

    distances, indices = nn.kneighbors(point)

    nearest_df = data.iloc[indices[0]].copy()
    nearest_df["distance"] = distances[0]

    return nearest_df

In [4]:
nearest_df = getNearestNeighbors(
    train,
    train.iloc[[0]],
    N = 25,
    soil_weight = 0.5,
    wilderness_weight = 0.5
)

NameError: name 'train' is not defined

In [27]:
def generateKnnPromptForPoint(X_train,y_train,test_point,N=25,filename="knn_prompt.txt"):

    dataset_size = len(X_train)

    assert(len(X_train) == len(y_train))
    assert(len(test_point)==1)

    nearest_df = getNearestNeighbors(
        X_train,
        test_point,
        N = N,
        soil_weight = 0.5,
        wilderness_weight = 0.5
    )

    nearest_df["class"] = y_train.loc[nearest_df.index]

    nearest_df = squashOneHots(nearest_df)
    nearest_df = truncateDecimals(nearest_df,digits=4)

    test_point = squashOneHots(test_point)
    test_point = truncateDecimals(test_point,digits=4)

    output_string = f"""
        You are tasked with identifying a tree's class / variety based on a variety of factors,
        including soil type, wilderness area, hillshade, distance to certain geological features, and local topography.

        You are given the features of one "test tree" (without the class label), as well as the features and classes of
        the top {N} trees (from a dataset of tens of thousands of trees), as measured by feature distance to the chosen test tree.

        Your task is to simply output one number, from 1 to 7, corresponding to the class you believe the tree belongs to.
        DO NOT simply perform a K-nearest-neighbors plurality. Use your world knowledge, general reasoning, and observations from the data
        to make an informed conclusion.

        {N} nearest neighbors:

        {nearest_df.to_csv()}

        Unclassified point:

        {test_point.to_csv()}

        Now, enter the class number (1 to 7) here. Your response should be a single number, with no additional text or formatting.
    """
    
    file = open(f"./prompts/{filename}","w")
    file.write(output_string)
    file.close()

    return output_string, nearest_df

In [6]:
generateKnnPromptForPoint(
    X_train,
    y_train,
    X_test.iloc[[0]],
    N=25,
    filename="knn_prompt.txt"
)

NameError: name 'X_train' is not defined

In [11]:
test_prompt = open("./prompts/knn_prompt.txt","r").read()

response = ollama.chat(
    model='gemma4:12b',
    messages=[{'role': 'user', 'content': test_prompt}],
    stream=False,
)

print(response['message']['content'])


KeyboardInterrupt: 

In [21]:
def getMode(df):
    class_col = df["class"]
    ls = class_col.mode().tolist()
    formatted = ",".join([str(int(x)) for x in ls])
    return ls[0],formatted

In [28]:
def fullKnnTest(X_train,y_train,X_test,y_test,N=25,model="gemma4:12b"):

    print(f"Proceeding with LLM-guided KNN test with N={N} neighbors, {len(X_train)} training examples, and {len(X_test)} test instances, using {model} via Ollama.")

    t0 = time.time()
    tL = time.time()

    iter_count = len(X_test)

    results = []

    for i in range(len(X_test)):
        print(f"Beginning iteration {i+1} of {iter_count} ({(i/iter_count)*100:.1f}% complete)...")

        test_point = X_test.iloc[[i]]

        prompt_filename = f"knn_prompt_{i}.txt"

        test_prompt, nearest_df = generateKnnPromptForPoint(
            X_train,
            y_train,
            test_point,
            N = N,
            filename = prompt_filename
        )

        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': test_prompt}],
            stream=False,
        )

        content = response['message']['content'].strip()

        print("CONTENT:")
        print(content)

        predicted_class = int(content) if content.isdigit() else -1
        if predicted_class > 7:
            predicted_class = -2

        true_class = int(y_test.iloc[i])

        mode_class,mode_str = getMode(nearest_df)

        results.append([predicted_class,true_class,mode_class,mode_str])

        ti = time.time()
        print(f"Completed iteration {i+1} in {ti-tL:.2f}s ({ti-t0:.2f}s total)")
        tL = ti

    column_labels = ["predicted_class","actual_class", "mode_class", "mode_str"]
    df = pd.DataFrame(results,columns=column_labels)

    print(f"Successfully completed all iterations in {time.time()-t0:.2f} seconds.")

    return df


In [29]:
train2,valid2,test2 = pd.read_csv("data/train.csv"),pd.read_csv("data/valid.csv"),pd.read_csv("data/test.csv")

train2 = downsample(train2, N=2000) # N is PER CLASS!!!!
valid2 = downsample(valid2, N=1000)
test2 = downsample(test2, N=1)

X_train2, y_train2 = train2.drop(columns="class"), train2["class"]
X_valid2, y_valid2 = valid2.drop(columns="class"), valid2["class"]
X_test2, y_test2 = test2.drop(columns="class"), test2["class"]

results = fullKnnTest(
    X_train2,
    y_train2,
    X_test2,
    y_test2,
    N=25,
    model="gemma4:12b"
)

results.head()

Proceeding with LLM-guided KNN test with N=25 neighbors, 13645 training examples, and 7 test instances, using gemma4:12b via Ollama.
Beginning iteration 1 of 7 (0.0% complete)...
CONTENT:
3
Completed iteration 1 in 8.95s (8.95s total)
Beginning iteration 2 of 7 (14.3% complete)...
CONTENT:

Completed iteration 2 in 15.57s (24.52s total)
Beginning iteration 3 of 7 (28.6% complete)...
CONTENT:

Completed iteration 3 in 14.13s (38.65s total)
Beginning iteration 4 of 7 (42.9% complete)...
CONTENT:

Completed iteration 4 in 14.77s (53.42s total)
Beginning iteration 5 of 7 (57.1% complete)...
CONTENT:
5
Completed iteration 5 in 13.65s (67.07s total)
Beginning iteration 6 of 7 (71.4% complete)...
CONTENT:

Completed iteration 6 in 16.14s (83.21s total)
Beginning iteration 7 of 7 (85.7% complete)...
CONTENT:

Completed iteration 7 in 14.26s (97.48s total)
Successfully completed all iterations in 97.48 seconds.


,predicted_class,actual_class,mode_class,mode_str
0,3,1,1.0,1
1,-1,2,2.0,2
2,-1,6,6.0,6
3,-1,3,3.0,3
4,5,5,5.0,5


In [30]:
print(results)

   predicted_class  actual_class  mode_class mode_str
0                3             1         1.0        1
1               -1             2         2.0        2
2               -1             6         6.0        6
3               -1             3         3.0        3
4                5             5         5.0        5
5               -1             4         4.0        4
6               -1             7         7.0        7
